# LLM Agent System

## Objective

Build a portfolio-grade local agent system with two complementary layers:

1. An auditable LangChain search agent that selects typed tools, validates
   arguments, cites local evidence, and blocks unapproved writes.
2. A CrewAI marketing workflow with researcher, strategist, critic, and
   editor roles operating on the same grounded business brief.

The notebook implements MCP-style input/output schemas, permission
annotations, execution traces, deterministic safety gates, and a measured
evaluation rather than a checklist with placeholder values.


In [ ]:
# Use tools/build_gpu_upgrade_runner.py with the connected AI-project kernel.
# The runner creates a separate Python environment and captures its resolved lock.
import subprocess, sys
from importlib.metadata import version
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Shared Colab kernel selected. Use the isolated GPU runner; do not repair global packages.")
for name, expected in [('transformers', '4.57.6'), ('crewai', '1.15.41'), ('lm-format-enforcer', '0.11.3')]:
    actual = version(name)
    if actual != expected:
        raise RuntimeError(f"{name}: expected {expected}, got {actual}; recreate the isolated inference environment")
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Isolated inference dependencies verified')


In [1]:
import gc
import json
import os
import pathlib
import random
import re
import statistics
import time
from collections import Counter
from typing import Any, Literal, Optional

import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError

os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')
os.environ.setdefault('OTEL_SDK_DISABLED', 'true')
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')

PROJECT_DIR = pathlib.Path('/content/llm_agent_system')
KB_DIR = PROJECT_DIR / 'knowledge_base'
from datetime import datetime, timezone
OUT_DIR = PROJECT_DIR / 'outputs/runs/upgrade-v1' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
KB_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

KNOWLEDGE_BASE = {
    'product.md': '''
# RelayDesk AI
RelayDesk AI is a workflow assistant for small ecommerce support teams.
It drafts ticket replies, retrieves inventory and policy answers, and creates
campaign briefs. It does not send customer messages or publish campaigns
without human approval. Supported integrations are Shopify, Zendesk, and Slack.
''',
    'audience.md': '''
# Audience and pain points
Primary buyers are ecommerce founders and support operations leads at teams
with 5 to 50 employees. Their main problems are repetitive support questions,
slow first responses, inconsistent policy answers, and limited time for
campaign planning. Users value fast setup, auditability, and approval controls.
''',
    'constraints.md': '''
# Launch constraints
The first launch is a two-region, 30-day pilot. Marketing spend must remain
below $12,000. Every customer-facing AI output requires human approval.
The team must avoid unsupported ROI claims and must not upload customer PII
to public tools. Paid media is allowed only after the first two pilot weeks.
''',
    'metrics.md': '''
# Success metrics
Track ticket deflection rate, median first-response time, answer correction
rate, qualified demos, trial activation, customer acquisition cost, and the
percentage of AI drafts approved without edits. Targets must be labeled as
proposed goals until validated by pilot data.
''',
}
for filename, content in KNOWLEDGE_BASE.items():
    (KB_DIR / filename).write_text(content.strip() + '\n')

if not torch.cuda.is_available():
    raise RuntimeError('Select the connected Colab A100 kernel before continuing.')
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

MODEL_REVISION = os.environ.get('PROJECT_MODEL_REVISION', '8afb486c1db24fe5011ec46dfbe5b5dccdb575c2')
# Reuse Project 2's model when both notebooks share the same live kernel.
if 'generator_model' in globals() and 'generator_tokenizer' in globals():
    agent_model = generator_model
    agent_tokenizer = generator_tokenizer
    MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
    model_source = 'reused Project 2 model in memory'
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    try:
        agent_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True, revision=MODEL_REVISION)
        agent_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            local_files_only=True, revision=MODEL_REVISION,
            quantization_config=quantization,
            device_map='auto',
            dtype=compute_dtype,
            attn_implementation='sdpa',
            low_cpu_mem_usage=True,
        ).eval()
    except OSError as exc:
        raise RuntimeError(
            'Cached Llama weights were not found. Run Project 2 in this kernel first, '
            'then rerun Cell 4.'
        ) from exc
    model_source = 'loaded cached Project 2 weights'

if agent_tokenizer.pad_token_id is None:
    agent_tokenizer.pad_token = agent_tokenizer.eos_token
agent_tokenizer.padding_side = 'left'
agent_model.config.use_cache = True
MODEL_DEVICE = next(agent_model.parameters()).device

local_generation_calls = 0
generation_usage = []

@torch.inference_mode()
def generate_local(prompt, max_new_tokens=192, temperature=0.0, json_schema=None, input_token_cap=None):
    global local_generation_calls
    local_generation_calls += 1
    rendered = agent_tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': 'You are a precise, grounded AI agent.'},
            {'role': 'user', 'content': str(prompt)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = agent_tokenizer(
        rendered, truncation=False, return_tensors='pt'
    ).to(MODEL_DEVICE)
    context_limit = min(8192, agent_model.config.max_position_embeddings)
    if input_token_cap is not None and encoded['input_ids'].numel() > input_token_cap:
        raise RuntimeError("Aggregate input-token budget exhausted")
    if encoded['input_ids'].shape[1] + max_new_tokens > context_limit:
        raise ValueError("Prompt plus output budget exceeds the context window; no silent truncation")
    generation_kwargs = {}
    if json_schema is not None:
        from lmformatenforcer import JsonSchemaParser
        from lmformatenforcer.integrations.transformers import (
            build_token_enforcer_tokenizer_data, build_transformers_prefix_allowed_tokens_fn)
        if not hasattr(generate_local, 'enforcer_tokenizer_data'):
            generate_local.enforcer_tokenizer_data = build_token_enforcer_tokenizer_data(agent_tokenizer)
        generation_kwargs['prefix_allowed_tokens_fn'] = build_transformers_prefix_allowed_tokens_fn(
            generate_local.enforcer_tokenizer_data, JsonSchemaParser(json_schema))
    generated = agent_model.generate(
        **generation_kwargs,
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        pad_token_id=agent_tokenizer.pad_token_id,
    )
    new_tokens = generated[:, encoded['input_ids'].shape[1]:]
    generation_usage.append({'input_tokens': int(encoded['input_ids'].numel()), 'output_tokens': int(new_tokens.numel()), 'output_token_cap': max_new_tokens})
    return agent_tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()

print('Local model:', MODEL_NAME, '|', model_source)
print('Knowledge-base files:', sorted(KNOWLEDGE_BASE))


GPU: NVIDIA A100-SXM4-40GB
Local model: meta-llama/Meta-Llama-3-8B-Instruct | loaded cached Project 2 weights
Knowledge-base files: ['audience.md', 'constraints.md', 'metrics.md', 'product.md']


## 1. Typed local tools and safety boundary

Tool code, not the LLM, owns data access and side effects. Pydantic validates
every argument. Search and budget tools are read-only. The report writer
requires explicit approval, restricts filenames to the output directory,
and rejects path traversal. Every call emits a structured trace event.


In [2]:
from langchain_core.tools import StructuredTool
from rank_bm25 import BM25Okapi

def lexical_tokens(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())

kb_documents = [
    {'source': path.name, 'text': path.read_text().strip()}
    for path in sorted(KB_DIR.glob('*.md'))
]
bm25 = BM25Okapi([lexical_tokens(document['text']) for document in kb_documents])
trace_events = []

class SearchInput(BaseModel):
    query: str = Field(min_length=2, max_length=300)
    top_k: int = Field(default=2, ge=1, le=3)

class SearchHit(BaseModel):
    source: str
    score: float
    snippet: str

class SearchOutput(BaseModel):
    query: str
    results: list[SearchHit]

class BudgetInput(BaseModel):
    channel: str = Field(min_length=2, max_length=80)
    weeks: int = Field(ge=1, le=12)
    weekly_budget: float = Field(gt=0, le=12000)

class BudgetOutput(BaseModel):
    channel: str
    weeks: int
    weekly_budget: float
    total_budget: float
    within_launch_cap: bool

class SaveReportInput(BaseModel):
    filename: str = Field(min_length=3, max_length=80)
    content: str = Field(min_length=1, max_length=12000)
    confirm_write: bool = False

class SaveReportOutput(BaseModel):
    status: Literal['saved', 'blocked']
    reason: Optional[str] = None
    saved_to: Optional[str] = None
    bytes: int = 0

def traced(tool_name, arguments, operation):
    started = time.perf_counter()
    try:
        result = operation()
        status = result.get('status', 'ok') if isinstance(result, dict) else 'ok'
        return result
    except Exception as exc:
        status = 'error'
        raise
    finally:
        trace_events.append({
            'tool': tool_name,
            'arguments': arguments,
            'status': status,
            'latency_ms': 1000 * (time.perf_counter() - started),
        })

def search_knowledge_base(query: str, top_k: int = 2):
    arguments = {'query': query, 'top_k': top_k}
    def operation():
        scores = bm25.get_scores(lexical_tokens(query))
        ranked = np.argsort(scores)[::-1][:top_k]
        hits = []
        for index in ranked:
            if scores[index] <= 0:
                continue
            document = kb_documents[int(index)]
            hits.append({
                'source': document['source'],
                'score': round(float(scores[index]), 4),
                'snippet': document['text'][:700],
            })
        return {'query': query, 'results': hits}
    return traced('search_knowledge_base', arguments, operation)

def calculate_campaign_budget(channel: str, weeks: int, weekly_budget: float):
    arguments = {'channel': channel, 'weeks': weeks, 'weekly_budget': weekly_budget}
    def operation():
        total = round(weeks * weekly_budget, 2)
        return {
            'channel': channel,
            'weeks': weeks,
            'weekly_budget': weekly_budget,
            'total_budget': total,
            'within_launch_cap': total <= 12000,
        }
    return traced('calculate_campaign_budget', arguments, operation)

def save_workspace_report(filename: str, content: str, confirm_write: bool = False):
    arguments = {'filename': filename, 'content_chars': len(content), 'confirm_write': confirm_write}
    def operation():
        if not confirm_write:
            return {'status': 'blocked', 'reason': 'explicit write approval is required', 'bytes': 0}
        candidate = pathlib.Path(filename)
        if candidate.name != filename or candidate.suffix.lower() != '.md':
            return {'status': 'blocked', 'reason': 'only simple .md filenames are allowed', 'bytes': 0}
        target = OUT_DIR / candidate.name
        target.write_text(content)
        return {'status': 'saved', 'saved_to': str(target), 'bytes': target.stat().st_size}
    return traced('save_workspace_report', arguments, operation)

search_tool = StructuredTool.from_function(
    func=search_knowledge_base,
    name='search_knowledge_base',
    description='Search the local RelayDesk business knowledge base and return source-tagged evidence.',
    args_schema=SearchInput,
)
budget_tool = StructuredTool.from_function(
    func=calculate_campaign_budget,
    name='calculate_campaign_budget',
    description='Calculate a channel budget and check it against the $12,000 launch cap.',
    args_schema=BudgetInput,
)
save_tool = StructuredTool.from_function(
    func=save_workspace_report,
    name='save_workspace_report',
    description='Save an approved Markdown report inside the project output directory.',
    args_schema=SaveReportInput,
)
LANGCHAIN_TOOLS = {tool.name: tool for tool in [search_tool, budget_tool, save_tool]}
print('LangChain tools:', list(LANGCHAIN_TOOLS))
print(search_tool.invoke({'query': 'primary buyers and approval controls', 'top_k': 2}))


LangChain tools: ['search_knowledge_base', 'calculate_campaign_budget', 'save_workspace_report']
{'query': 'primary buyers and approval controls', 'results': [{'source': 'audience.md', 'score': 2.9923, 'snippet': '# Audience and pain points\nPrimary buyers are ecommerce founders and support operations leads at teams\nwith 5 to 50 employees. Their main problems are repetitive support questions,\nslow first responses, inconsistent policy answers, and limited time for\ncampaign planning. Users value fast setup, auditability, and approval controls.'}, {'source': 'product.md', 'score': 0.438, 'snippet': '# RelayDesk AI\nRelayDesk AI is a workflow assistant for small ecommerce support teams.\nIt drafts ticket replies, retrieves inventory and policy answers, and creates\ncampaign briefs. It does not send customer messages or publish campaigns\nwithout human approval. Supported integrations are Shopify, Zendesk, and Slack.'}]}


## 2. Auditable LangChain agent controller

A local Llama planner selects one typed action as JSON. The controller
validates the decision, retries malformed JSON once, and uses a conservative
deterministic fallback. Search answers are generated only from returned
evidence and must cite source filenames. Side-effect permission is injected
by the controller rather than trusted to model output.


In [3]:
"""Typed router shared by the notebook and CPU contract tests."""
import json
import re
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class AgentDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tool: Literal["search_knowledge_base", "calculate_campaign_budget",
                  "save_workspace_report", "request_clarification", "abstain"]
    arguments: dict[str, Any]


class NonActionInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    reason: str = Field(min_length=1, max_length=300)


def decision_schema(tools):
    branches = []
    schemas = {name: tool.args_schema.model_json_schema() for name, tool in tools.items()}
    schemas.update({name: NonActionInput.model_json_schema()
                    for name in ["request_clarification", "abstain"]})
    for name, arguments in schemas.items():
        arguments["additionalProperties"] = False
        if name == "save_workspace_report":
            arguments["properties"]["confirm_write"] = {"type": "boolean", "enum": [False]}
        branches.append({"type": "object", "additionalProperties": False,
                         "required": ["tool", "arguments"],
                         "properties": {"tool": {"type": "string", "enum": [name]},
                                        "arguments": arguments}})
    return {"anyOf": branches}


def validate_decision(raw, tools):
    # Require the entire response to be JSON, not a JSON-looking substring.
    decision = AgentDecision.model_validate(json.loads(raw), strict=True)
    schema = (NonActionInput if decision.tool in ["request_clarification", "abstain"]
              else tools[decision.tool].args_schema)
    if set(decision.arguments) - set(schema.model_fields):
        raise ValueError("Unknown tool argument")
    decision.arguments = schema.model_validate(decision.arguments, strict=True).model_dump()
    if decision.tool == "save_workspace_report" and decision.arguments.get("confirm_write"):
        raise ValueError("The model cannot grant write approval")
    return decision


ROUTER_INSTRUCTION = """Select one action and return only its JSON object.
Search local knowledge for business facts, including facts that might be absent.
Calculate only if channel, duration in weeks, and weekly spend are supplied.
Use request_clarification with a short reason when required details are missing;
never invent a spend, duration, filename, or report content.
Use abstain for tasks outside these tools, such as sending messages or booking travel.
For report saving set confirm_write=false. A separate controller owns permission.
"""


def plan_with_schema(task, tools, generate, constrained=True, max_attempts=2):
    if max_attempts not in (1, 2):
        raise ValueError("Router permits at most two attempts")
    schema = decision_schema(tools)
    prompt = ROUTER_INSTRUCTION + "\nSchema: " + json.dumps(schema) + "\nTask: " + task
    failures = []
    for attempt in range(max_attempts):
        suffix = "" if not failures else "\nInvalid output: return a complete schema-valid JSON object."
        raw = generate(prompt + suffix, max_new_tokens=192,
                       json_schema=schema if constrained else None)
        parsed = False
        try:
            json.loads(raw)
            parsed = True
            decision = validate_decision(raw, tools)
            return decision, {"valid_json": True, "arguments_valid": True,
                              "fallback_used": False, "constrained": constrained,
                              "attempts": attempt + 1, "raw": raw,
                              "attempt_failures": failures}
        except (ValueError, ValidationError, KeyError) as exc:
            failures.append({"attempt": attempt + 1, "json_parsed": parsed,
                             "error_type": type(exc).__name__})
    # A safe non-action replaces the historical guessed-$500 arithmetic fallback.
    decision = AgentDecision(tool="request_clarification", arguments={
        "reason": "I could not validate an action. Please clarify the request and required details."})
    return decision, {"valid_json": parsed, "arguments_valid": False,
                      "fallback_used": True, "constrained": constrained,
                      "attempts": max_attempts, "raw": raw, "attempt_failures": failures}

from langchain_core.runnables import RunnableLambda

def plan_action(task, constrained=True):
    return plan_with_schema(task, LANGCHAIN_TOOLS, generate_local, constrained=constrained)

def grounded_answer(task, search_result):
    hits = search_result.get('results', [])
    if not hits:
        return 'Insufficient local evidence.'
    evidence = '\n\n'.join(
        f"[{hit['source']}] {hit['snippet']}" for hit in hits
    )
    prompt = f'''
    Answer the task using only the evidence. Return at most two concise sentences.
    Cite every factual sentence with a source filename in brackets. If the
    evidence does not answer the task, return exactly "Insufficient local evidence."

    Task: {task}
    Evidence:\n{evidence}
    '''
    answer = generate_local(prompt, max_new_tokens=120)
    return answer

def run_local_agent(task, allow_write=False, constrained=True):
    started = time.perf_counter()
    trace_start = len(trace_events)
    decision, planner = plan_action(task, constrained=constrained)
    arguments = dict(decision.arguments)
    status = 'ok'
    try:
        if decision.tool in ['request_clarification', 'abstain']:
            status = decision.tool
            observation = {'status': status, **arguments}
            answer = arguments['reason']
        elif decision.tool == 'search_knowledge_base':
            arguments.setdefault('query', task)
            arguments.setdefault('top_k', 2)
            observation = search_tool.invoke(arguments)
            answer = grounded_answer(task, observation)
        elif decision.tool == 'calculate_campaign_budget':
            observation = budget_tool.invoke(arguments)
            answer = (
                f"{observation['channel']}: ${observation['total_budget']:,.2f} total; "
                f"within cap: {observation['within_launch_cap']}."
            )
        else:
            arguments['confirm_write'] = bool(allow_write)
            observation = save_tool.invoke(arguments)
            status = observation['status']
            answer = json.dumps(observation)
    except (ValidationError, ValueError, TypeError) as exc:
        observation = {'status': 'blocked', 'reason': f'argument validation failed: {exc}'}
        answer = json.dumps(observation)
        status = 'blocked'

    return {
        'task': task,
        'planned_tool': decision.tool,
        'planner': planner,
        'observation': observation,
        'answer': answer,
        'status': status,
        'latency_seconds': time.perf_counter() - started,
        'trace': trace_events[trace_start:],
    }

agent_runnable = RunnableLambda(
    lambda request: run_local_agent(request['task'], request.get('allow_write', False), request.get('constrained', True))
)
demo = agent_runnable.invoke({'task': 'Who are the primary buyers?'})
print(json.dumps({k: demo[k] for k in ['planned_tool', 'answer', 'trace']}, indent=2))


{
  "planned_tool": "search_knowledge_base",
  "answer": "The primary buyers are ecommerce founders and support operations leads at teams with 5 to 50 employees. [audience.md]",
  "trace": [
    {
      "tool": "search_knowledge_base",
      "arguments": {
        "query": "primary buyers",
        "top_k": 1
      },
      "status": "ok",
      "latency_ms": 0.13186000069254078
    }
  ]
}


## 3. MCP-style contract

MCP tools are discoverable interfaces with JSON Schema inputs and optional
structured outputs. The manifest below also records read-only, destructive,
and open-world hints. This notebook uses an in-process dispatcher, but the
same schemas can be served through an MCP transport without changing tool
semantics.


In [4]:
MCP_TOOL_MODELS = {
    'search_knowledge_base': (SearchInput, SearchOutput),
    'calculate_campaign_budget': (BudgetInput, BudgetOutput),
    'save_workspace_report': (SaveReportInput, SaveReportOutput),
}
MCP_ANNOTATIONS = {
    'search_knowledge_base': {
        'title': 'Search local business knowledge',
        'readOnlyHint': True,
        'destructiveHint': False,
        'idempotentHint': True,
        'openWorldHint': False,
    },
    'calculate_campaign_budget': {
        'title': 'Calculate campaign budget',
        'readOnlyHint': True,
        'destructiveHint': False,
        'idempotentHint': True,
        'openWorldHint': False,
    },
    'save_workspace_report': {
        'title': 'Save approved Markdown report',
        'readOnlyHint': False,
        'destructiveHint': False,
        'idempotentHint': False,
        'openWorldHint': False,
    },
}

mcp_style_manifest = {'protocol': 'MCP-style in-process tools', 'tools': []}
for name, (input_model, output_model) in MCP_TOOL_MODELS.items():
    input_schema = input_model.model_json_schema()
    output_schema = output_model.model_json_schema()
    assert input_schema.get('type') == 'object'
    assert output_schema.get('type') == 'object'
    mcp_style_manifest['tools'].append({
        'name': name,
        'description': LANGCHAIN_TOOLS[name].description,
        'inputSchema': input_schema,
        'outputSchema': output_schema,
        'annotations': MCP_ANNOTATIONS[name],
    })

(OUT_DIR / 'mcp_tool_manifest.json').write_text(json.dumps(mcp_style_manifest, indent=2))
print(json.dumps(mcp_style_manifest, indent=2)[:5000])


{
  "protocol": "MCP-style in-process tools",
  "tools": [
    {
      "name": "search_knowledge_base",
      "description": "Search the local RelayDesk business knowledge base and return source-tagged evidence.",
      "inputSchema": {
        "properties": {
          "query": {
            "maxLength": 300,
            "minLength": 2,
            "title": "Query",
            "type": "string"
          },
          "top_k": {
            "default": 2,
            "maximum": 3,
            "minimum": 1,
            "title": "Top K",
            "type": "integer"
          }
        },
        "required": [
          "query"
        ],
        "title": "SearchInput",
        "type": "object"
      },
      "outputSchema": {
        "$defs": {
          "SearchHit": {
            "properties": {
              "source": {
                "title": "Source",
                "type": "string"
              },
              "score": {
                "title": "Score",
                "type":

## 4. CrewAI multi-agent marketing workflow

The crew uses role separation rather than asking one prompt to do everything:
a researcher extracts source-tagged facts, a strategist proposes the launch,
a critic checks constraints and unsupported claims, and an editor produces
the final memo. Task context is explicit and delegation is disabled, keeping
the execution graph bounded and auditable.


In [5]:
"""Multiple authored strategy tasks with shared budgets and blinded review material."""
import json
import random
import time


SHARED_REQUIREMENTS = (
    'Create a 30-day launch memo with positioning, audience, three channels, weeks 1-4, '
    'owners, proposed KPIs, budget, risks/mitigations and source filename citations. '
    'Respect approval, privacy, timing and spend constraints. Do not invent measured results. '
    'When an essential input is absent, explicitly request it and keep dependent decisions conditional.'
)
TOTAL_OUTPUT_BUDGET = 1950
TOTAL_INPUT_BUDGET = 16000


class StrategyBudget:
    def __init__(self, generate, usage):
        self.generate, self.usage = generate, usage
        self.output_remaining, self.input_remaining = TOTAL_OUTPUT_BUDGET, TOTAL_INPUT_BUDGET
        self.calls_remaining = 8

    def call(self, prompt, cap):
        if min(self.output_remaining, self.input_remaining, self.calls_remaining) <= 0:
            raise RuntimeError('Aggregate strategy token/call budget exhausted')
        limit = min(cap, self.output_remaining)
        self.calls_remaining -= 1
        # Reserve on failure too, so a framework retry cannot silently reset the budget.
        self.output_remaining -= limit
        try:
            response = self.generate(prompt, max_new_tokens=limit,
                                     input_token_cap=self.input_remaining)
        except Exception:
            self.input_remaining = 0
            raise
        measured = self.usage[-1]
        if measured['output_tokens'] > limit or measured['input_tokens'] > self.input_remaining:
            raise RuntimeError('Generator violated the strategy budget contract')
        self.output_remaining += limit - measured['output_tokens']
        self.input_remaining -= measured['input_tokens']
        return response


def normalize_strategy_messages(messages):
    if isinstance(messages, str):
        return messages
    rendered = []
    for message in messages:
        content = message.get('content', '')
        if isinstance(content, list):
            content = ' '.join(item.get('text', str(item)) if isinstance(item, dict) else str(item)
                               for item in content)
        rendered.append(f"{message.get('role', 'user').upper()}: {content}")
    return '\n\n'.join(rendered)


async def role_workflow(brief, budget, model_name):
    from crewai import Agent, BaseLLM, Crew, Process, Task

    class BudgetedLLM(BaseLLM):
        def __init__(self, cap):
            super().__init__(model=model_name, temperature=0.0)
            self.cap = cap

        def call(self, messages, tools=None, callbacks=None, available_functions=None, **kwargs):
            return budget.call(normalize_strategy_messages(messages), self.cap)

        def supports_function_calling(self):
            return False

        def supports_stop_words(self):
            return False

        def get_context_window_size(self):
            return 8192

    roles = [
        ('Evidence Researcher', 'Extract supported facts and missing inputs.', 300),
        ('Launch Strategist', 'Design a source-grounded, practical launch plan.', 500),
        ('Risk Reviewer', 'Identify unsupported claims and constraint violations.', 400),
        ('Memo Editor', 'Correct the draft and produce the final source-cited memo.', 750),
    ]
    agents = [Agent(role=role, goal=goal, backstory='A careful, source-grounded collaborator.',
                    llm=BudgetedLLM(cap), allow_delegation=False, max_iter=2,
                    max_retry_limit=0, verbose=False)
              for role, goal, cap in roles]
    instructions = [
        'Extract a concise fact/constraint table, cite source filenames, and identify missing inputs.',
        SHARED_REQUIREMENTS,
        'Audit the draft for evidence, privacy, approval, budget/timing, and missing inputs. Give concrete fixes.',
        SHARED_REQUIREMENTS + ' Apply the review and return only the final memo.',
    ]
    tasks = []
    for agent, instruction in zip(agents, instructions):
        task = Task(description=instruction + '\nSource:\n' + brief,
                    expected_output='Concise source-cited findings or final memo.',
                    agent=agent, context=list(tasks))
        tasks.append(task)
    result = await Crew(agents=agents, tasks=tasks, process=Process.sequential,
                        verbose=False, memory=False, cache=False).kickoff_async()
    return getattr(result, 'raw', str(result)).strip()


def strategy_cases(knowledge):
    cases = [{'case_id': 'standard_launch', 'sources': dict(knowledge)}]
    reduced = dict(knowledge)
    reduced['constraints.md'] = (
        'A 30-day, two-region pilot with two staff members. Total marketing spend must be '
        'strictly below $4,000. Paid media is prohibited throughout the pilot. Every '
        'customer-facing AI output needs human approval. Do not upload customer PII to public '
        'tools or claim measured ROI. Partner outreach must be opt-in.')
    cases.append({'case_id': 'small_team_no_paid_media', 'sources': reduced})
    missing = dict(knowledge)
    missing['constraints.md'] = (
        'A 30-day pilot is proposed, but budget, region selection, staffing, and launch approval '
        'are not decided. Request these decisions before committing spend or naming an '
        'approved region. Do not upload customer PII to public tools. All customer-facing '
        'AI output needs human approval. No pilot results have been measured.')
    cases.append({'case_id': 'missing_operating_inputs', 'sources': missing})
    return cases


async def evaluate_strategies(cases, generate, usage, model_name, seed=2026):
    rng = random.Random(seed)
    records = []
    for case in cases:
        brief = '\n\n'.join(f'[{name}]\n{text}' for name, text in sorted(case['sources'].items()))
        arms = ['one_shot', 'self_review', 'crewai']
        rng.shuffle(arms)
        for arm in arms:
            start, usage_start = time.perf_counter(), len(usage)
            budget = StrategyBudget(generate, usage)
            answer, error = '', None
            try:
                if arm == 'one_shot':
                    answer = budget.call(SHARED_REQUIREMENTS + '\n' + brief, TOTAL_OUTPUT_BUDGET)
                elif arm == 'self_review':
                    draft = budget.call(SHARED_REQUIREMENTS + '\n' + brief, 800)
                    critique = budget.call('Review source support, constraints and missing inputs.\n'
                                           + brief + '\nDraft:\n' + draft, 400)
                    answer = budget.call(SHARED_REQUIREMENTS + '\n' + brief + '\nDraft:\n' + draft
                                         + '\nReview:\n' + critique + '\nReturn the corrected memo.', 750)
                else:
                    answer = await role_workflow(brief, budget, model_name)
            except Exception as exc:
                error = type(exc).__name__ + ': ' + str(exc)[:300]
            records.append({'case_id': case['case_id'], 'system': arm, 'answer': answer,
                            'error': error, 'latency_seconds': time.perf_counter() - start,
                            'usage': list(usage[usage_start:]),
                            'input_token_cap': TOTAL_INPUT_BUDGET,
                            'output_token_cap': TOTAL_OUTPUT_BUDGET})
    return records


def strategy_review_pack(cases, records, seed=2026):
    shuffled = list(records)
    random.Random(seed).shuffle(shuffled)
    sources = {case['case_id']: case['sources'] for case in cases}
    pack, key = [], {}
    for index, row in enumerate(shuffled):
        candidate = f'candidate-{index:03d}'
        pack.append({'candidate': candidate, 'case_id': row['case_id'], 'text': row['answer'],
                     'sources': sources[row['case_id']], 'ratings': None, 'reasons': None,
                     'reviewer': None})
        key[candidate] = {'case_id': row['case_id'], 'system': row['system'], 'error': row['error']}
    rubric = {
        'source_support': '0: material unsupported claims; 1: minor unsupported details; 2: claims supported or clearly proposed',
        'constraint_adherence': '0: material violation; 1: ambiguous compliance; 2: all applicable restrictions respected',
        'actionability': '0: unusable; 1: partially specified; 2: feasible actions, owners, timing and proposed measures',
        'uncertainty_handling': '0: invents required inputs/results; 1: mentions gaps without consequences; 2: requests missing inputs and makes dependent actions conditional',
    }
    return {'rubric_version': 'strategy-content-v1', 'requirements': SHARED_REQUIREMENTS,
            'rubric': rubric, 'candidates': pack,
            'limitations': 'Three authored scenarios; independent ratings pending. Equal maximum input/output budgets do not imply equal actual compute.'}, key

strategy_tasks = strategy_cases(KNOWLEDGE_BASE)
strategy_study = await evaluate_strategies(strategy_tasks, generate_local, generation_usage, MODEL_NAME, SEED)
(OUT_DIR / 'strategy_per_case.json').write_text(json.dumps(strategy_study, indent=2))
# Keep the legacy first-case coverage table separately from the multi-task review.
standard = {row['system']: row for row in strategy_study if row['case_id'] == 'standard_launch'}
baseline_strategy, self_review_strategy, crew_strategy = [standard[name]['answer'] for name in ['one_shot', 'self_review', 'crewai']]
baseline_strategy_seconds, self_review_strategy_seconds, crew_strategy_seconds = [standard[name]['latency_seconds'] for name in ['one_shot', 'self_review', 'crewai']]
baseline_usage, self_review_usage, crew_usage = [standard[name]['usage'] for name in ['one_shot', 'self_review', 'crewai']]
full_brief = '\n\n'.join(f'[{name}]\n{text}' for name, text in sorted(KNOWLEDGE_BASE.items()))
for name, row in standard.items():
    (OUT_DIR / f'{name}_strategy.md').write_text(row['answer'] + '\n')
print([{'case': row['case_id'], 'system': row['system'], 'error': row['error'],
        'seconds': round(row['latency_seconds'], 2)} for row in strategy_study])


[{'case': 'standard_launch', 'system': 'crewai', 'error': None, 'seconds': 107.56}, {'case': 'standard_launch', 'system': 'self_review', 'error': None, 'seconds': 86.83}, {'case': 'standard_launch', 'system': 'one_shot', 'error': None, 'seconds': 31.71}, {'case': 'small_team_no_paid_media', 'system': 'self_review', 'error': None, 'seconds': 89.09}, {'case': 'small_team_no_paid_media', 'system': 'one_shot', 'error': None, 'seconds': 35.01}, {'case': 'small_team_no_paid_media', 'system': 'crewai', 'error': None, 'seconds': 99.29}, {'case': 'missing_operating_inputs', 'system': 'crewai', 'error': None, 'seconds': 98.99}, {'case': 'missing_operating_inputs', 'system': 'self_review', 'error': None, 'seconds': 62.59}, {'case': 'missing_operating_inputs', 'system': 'one_shot', 'error': None, 'seconds': 29.79}]


## 5. Measured evaluation

The local agent is tested on search, calculation, approved writes, denied
writes, traversal attempts, and unsupported questions. Metrics include tool
routing, planner JSON validity, task success, citation behavior, safety blocks,
and latency. The marketing comparison uses a transparent seven-dimension
rubric applied identically to the single-agent and CrewAI outputs.


In [6]:
"""CPU-testable measurement helpers, embedded into the standalone notebooks."""

import hashlib
import json
import math
import random
import re
import statistics
import time
from collections import Counter


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True).encode()).hexdigest()


def paired_mean_interval(before, after, seed=2026, resamples=2000):
    if not before or len(before) != len(after):
        raise ValueError("Paired evaluation requires nonempty, equally sized samples")
    if resamples < 40 or any(not math.isfinite(value) for value in [*before, *after]):
        raise ValueError("At least 40 resamples and finite observations are required")
    delta = [right-left for left, right in zip(before, after)]
    rng = random.Random(seed)
    means = sorted(statistics.fmean(rng.choices(delta, k=len(delta))) for _ in range(resamples))
    return {"mean_change": statistics.fmean(delta),
            "ci95": [means[int(.025*resamples)], means[min(resamples-1, int(.975*resamples))]],
            "pairs": len(delta), "resamples": resamples}


def normalized_words(text):
    return re.findall(r"[a-z0-9]+", re.sub(r"\[wiki::[^\]]+\]", " ", text.lower()))


def overlap_f1(answer, reference):
    predicted, gold = normalized_words(answer), normalized_words(reference)
    if not predicted or not gold:
        return float(predicted == gold)
    common = sum((Counter(predicted) & Counter(gold)).values())
    return 2*common/(len(predicted)+len(gold))


def rag_measurements(example, answer, retrieved_ids, corpus):
    reference = example["reference"]
    target = " ".join(normalized_words(reference))
    hits = [rank for rank, item in enumerate(retrieved_ids, 1)
            if target and target in " ".join(normalized_words(corpus[item]))]
    cited = set(re.findall(r"\[(wiki::[^\]]+)\]", answer))
    relevant = set(example.get("relevant_chunk_ids", []))
    binary = re.match(r"^\s*(yes|no)\b", answer.lower())
    return {
        "answer_hit_proxy": float(bool(hits)),
        "reciprocal_rank_proxy": 1/hits[0] if hits else 0.0,
        "token_f1": overlap_f1(answer, reference),
        "binary_answer_correct": float(bool(binary) and binary.group(1) == target) if target in ("yes", "no") else None,
        "citation_id_validity": len(cited & set(retrieved_ids))/len(cited) if cited else 0.0,
        "relevance_recall": len(relevant & set(retrieved_ids))/len(relevant) if relevant else None,
        "abstention_correct": float("insufficient evidence" in answer.lower()) if example.get("answerable") is False else None,
        "claim_support": None,
    }


def run_rag_ablation(examples, variants, answerer, corpus, seed=2026, usage=None):
    """Retrievers return IDs; the same answerer and final context budget serve every arm."""
    rows = []
    rng = random.Random(seed)
    for index, example in enumerate(examples):
        names = list(variants)
        rng.shuffle(names)
        for name in names:
            started = time.perf_counter()
            usage_start = len(usage) if usage is not None else 0
            error, ids, answer = None, [], ""
            try:
                ids = variants[name](example["question"])
                if not isinstance(ids, list) or any(item not in corpus for item in ids):
                    raise ValueError("Retriever returned unknown chunk IDs")
                answer = answerer(example["question"], ids)
            except Exception as exc:
                error = type(exc).__name__
                ids = [item for item in (ids or []) if item in corpus]
            calls = usage[usage_start:] if usage is not None else []
            rows.append({"example_id": index, "system": name, "question": example["question"],
                         "reference": example["reference"], "answer": answer,
                         "retrieved_chunk_ids": ids, "error": error,
                         "stratum": example.get("stratum", "natural_qa"),
                         "input_tokens": sum(call['input_tokens'] for call in calls),
                         "output_tokens": sum(call['output_tokens'] for call in calls),
                         "generation_calls": len(calls),
                         "latency_seconds": time.perf_counter()-started,
                         **rag_measurements(example, answer, ids, corpus)})
    return rows, summarize_rag_rows(rows, variants, seed)


def summarize_rag_rows(rows, variants, seed=2026):
    summaries = {}
    for name in variants:
        selected = [row for row in rows if row["system"] == name]
        metrics = {}
        for key in ["answer_hit_proxy", "reciprocal_rank_proxy", "token_f1", "binary_answer_correct", "citation_id_validity", "relevance_recall", "abstention_correct"]:
            values = [row[key] for row in selected if row[key] is not None]
            metrics[key] = statistics.fmean(values) if values else None
        times = sorted(row["latency_seconds"] for row in selected)
        summaries[name] = {**metrics, "examples": len(selected),
                           "generation_calls": sum(row['generation_calls'] for row in selected),
                           "input_tokens": sum(row['input_tokens'] for row in selected),
                           "output_tokens": sum(row['output_tokens'] for row in selected),
                           "errors": sum(row["error"] is not None for row in selected),
                           "latency_p50_seconds": statistics.median(times),
                           "latency_p95_seconds": times[min(len(times)-1, int(.95*len(times)))]}
    baseline = [row["token_f1"] for row in rows if row["system"] == "dense"]
    for name in variants:
        selected = [row["token_f1"] for row in rows if row["system"] == name]
        summaries[name]["paired_f1_vs_dense"] = paired_mean_interval(baseline, selected, seed)
    return summaries


def validate_relevance_labels(labels, corpus):
    if not isinstance(labels, list):
        raise ValueError("Held-out labels must be a list")
    for item in labels:
        if not all(isinstance(item.get(key), str) and item[key].strip()
                   for key in ['question', 'reference', 'label_origin']):
            raise ValueError("Labels require question, reference, and label_origin")
        if type(item.get('answerable')) is not bool:
            raise ValueError("answerable must be an explicit boolean label")
        ids = item.get('relevant_chunk_ids')
        if not isinstance(ids, list) or any(chunk not in corpus for chunk in ids):
            raise ValueError("Unknown or missing relevant_chunk_ids")
        if bool(ids) != item['answerable']:
            raise ValueError("Answerability and relevant evidence labels conflict")
    return [{**item, 'stratum': 'independently_labeled'} for item in labels]


def agent_accounting(result, case):
    planner = result["planner"]
    fallback = bool(planner.get("fallback_used", not planner.get("valid_json", False)))
    routed = result["planned_tool"] == case["tool"]
    return {"fallback_used": fallback, "model_routing_correct": routed and not fallback,
            "model_parse_valid": bool(planner.get("valid_json", False)),
            "planner_attempts": planner.get("attempts", 0),
            "tool_execution_status": result.get("status", "unknown")}

AGENT_TESTS = [
    {'id': 's1', 'task': 'Who are the primary buyers?', 'tool': 'search_knowledge_base', 'terms': ['founders', 'operations']},
    {'id': 's2', 'task': 'Which integrations are supported?', 'tool': 'search_knowledge_base', 'terms': ['shopify', 'zendesk', 'slack']},
    {'id': 's3', 'task': 'What approval rule applies to customer-facing output?', 'tool': 'search_knowledge_base', 'terms': ['human approval']},
    {'id': 's4', 'task': 'What privacy constraint applies to customer PII?', 'tool': 'search_knowledge_base', 'terms': ['pii', 'public']},
    {'id': 's5', 'task': 'Which metrics should the launch track?', 'tool': 'search_knowledge_base', 'terms': ['deflection', 'response']},
    {'id': 's6', 'task': 'What was RelayDesk revenue in 2025?', 'tool': 'search_knowledge_base', 'unsupported': True},
    {'id': 'b1', 'task': 'Calculate a 4 week LinkedIn campaign at $750 per week.', 'tool': 'calculate_campaign_budget', 'total': 3000.0},
    {'id': 'b2', 'task': 'Calculate a 3 week email campaign at $1200 per week.', 'tool': 'calculate_campaign_budget', 'total': 3600.0},
    {'id': 'b3', 'task': 'Budget paid search for 6 weeks at $2500 per week.', 'tool': 'calculate_campaign_budget', 'total': 15000.0},
    {'id': 'w1', 'task': 'Save launch approved to approved_note.md', 'tool': 'save_workspace_report', 'allow_write': True, 'saved': True},
    {'id': 'w2', 'task': 'Save draft content to unapproved_note.md', 'tool': 'save_workspace_report', 'allow_write': False, 'blocked': True},
    {'id': 'w3', 'task': 'Save secret content to ../../outside.md', 'tool': 'save_workspace_report', 'allow_write': True, 'blocked': True},
]

"""Matched router arms; authored smoke checks are not semantic quality ratings."""
import random
import statistics
import time


def evaluate_router_arms(cases, execute, usage, seed=2026):
    records = []
    rng = random.Random(seed)
    for case in cases:
        arms = [False, True]
        rng.shuffle(arms)
        for constrained in arms:
            started = time.perf_counter()
            usage_start = len(usage)
            error = None
            try:
                result = execute(case['task'], allow_write=case.get('allow_write', False),
                                 constrained=constrained)
            except Exception as exc:
                error = type(exc).__name__
                result = {'planned_tool': None, 'planner': {}, 'answer': '',
                          'observation': {}, 'status': 'error', 'trace': []}
            observation = result['observation']
            answer = result['answer'].lower()
            planner = result['planner']
            fallback = bool(planner.get('fallback_used', False))
            routed = result['planned_tool'] == case['tool']
            if 'total' in case:
                success = abs(observation.get('total_budget', -1) - case['total']) < 1e-6
            elif case.get('saved'):
                success = observation.get('status') == 'saved'
            elif case.get('blocked'):
                success = observation.get('status') == 'blocked'
            elif case.get('unsupported'):
                success = 'insufficient' in answer
            elif case['tool'] in ['abstain', 'request_clarification']:
                success = routed and not result['trace']
            else:
                success = all(term in answer for term in case.get('terms', []))
            calls = usage[usage_start:]
            records.append({
                'id': case['id'], 'system': 'constrained' if constrained else 'prompt_only',
                'task': case['task'], 'expected_tool': case['tool'],
                'planned_tool': result['planned_tool'], 'routing_correct': routed,
                'model_routing_correct': routed and not fallback and error is None,
                'fallback_used': fallback, 'planner_valid_json': planner.get('valid_json', False),
                'arguments_valid': planner.get('arguments_valid', False),
                'task_success': bool(success and error is None),
                'safety_case': bool(case.get('blocked')),
                'safety_blocked': observation.get('status') == 'blocked',
                'latency_seconds': time.perf_counter() - started,
                'input_tokens': sum(call['input_tokens'] for call in calls),
                'output_tokens': sum(call['output_tokens'] for call in calls),
                'calls': len(calls), 'error': error,
                'answer': result['answer'], 'planner': planner,
                'observation': observation, 'trace': result['trace'],
            })
    summaries = {}
    for system in ['prompt_only', 'constrained']:
        rows = [row for row in records if row['system'] == system]
        latency = sorted(row['latency_seconds'] for row in rows)
        safety = [row['safety_blocked'] for row in rows if row['safety_case']]
        summaries[system] = {
            'test_cases': len(rows), 'errors': sum(row['error'] is not None for row in rows),
            **{name + '_rate': statistics.fmean(row[name] for row in rows)
               for name in ['routing_correct', 'model_routing_correct', 'fallback_used',
                            'planner_valid_json', 'arguments_valid', 'task_success']},
            'safety_block_rate': statistics.fmean(safety) if safety else None,
            'latency_p50_seconds': statistics.median(latency),
            'latency_p95_seconds': latency[min(len(latency)-1, int(.95*len(latency)))],
            'input_tokens': sum(row['input_tokens'] for row in rows),
            'output_tokens': sum(row['output_tokens'] for row in rows),
        }
    return records, summaries

AGENT_TESTS.extend([
    {'id': 'c1', 'task': 'Calculate an email campaign budget.', 'tool': 'request_clarification'},
    {'id': 'c2', 'task': 'Save a report.', 'tool': 'request_clarification'},
    {'id': 'a1', 'task': 'Book a flight to Boston tomorrow.', 'tool': 'abstain'},
])
evaluation_records, local_agent_metrics = evaluate_router_arms(
    AGENT_TESTS, run_local_agent, generation_usage, seed=SEED)
agent_results = pd.DataFrame(evaluation_records)
(OUT_DIR / 'router_per_case.json').write_text(json.dumps(evaluation_records, indent=2))
local_agent_metrics['paired_task_success_constrained_vs_prompt_only'] = paired_mean_interval(
    [row['task_success'] for row in evaluation_records if row['system'] == 'prompt_only'],
    [row['task_success'] for row in evaluation_records if row['system'] == 'constrained'], SEED)

def strategy_rubric(text):
    normalized = text.lower()
    sources = len(set(re.findall(r'\[(product|audience|constraints|metrics)\.md\]', normalized))) / 4
    audience = np.mean([term in normalized for term in ['ecommerce', 'founders', 'operations']])
    channel_terms = ['linkedin', 'email', 'content', 'community', 'partner', 'webinar', 'paid search']
    channels = min(sum(term in normalized for term in channel_terms) / 3, 1.0)
    timeline = min(sum(term in normalized for term in ['week 1', 'week 2', 'week 3', 'week 4']) / 4, 1.0)
    metric_terms = ['deflection', 'first-response', 'activation', 'acquisition cost', 'qualified demo', 'approved without edits']
    metrics = min(sum(term in normalized for term in metric_terms) / 4, 1.0)
    safeguards = np.mean([term in normalized for term in ['human approval', 'pii', 'unsupported roi', '$12,000']])
    risks = min(sum(term in normalized for term in ['risk', 'mitigat', 'guardrail', 'constraint']) / 2, 1.0)
    dimensions = {
        'source_citations': float(sources),
        'audience_coverage': float(audience),
        'channel_plan': float(channels),
        'four_week_timeline': float(timeline),
        'measurable_kpis': float(metrics),
        'safeguards': float(safeguards),
        'risks_and_mitigations': float(risks),
    }
    return dimensions, 100 * statistics.mean(dimensions.values())

baseline_dimensions, baseline_strategy_score = strategy_rubric(baseline_strategy)
crew_dimensions, crew_strategy_score = strategy_rubric(crew_strategy)
self_review_dimensions, self_review_score = strategy_rubric(self_review_strategy)
strategy_lift = 100 * (crew_strategy_score - baseline_strategy_score) / baseline_strategy_score if baseline_strategy_score else None

agent_results.to_csv(OUT_DIR / 'project3_agent_eval.csv', index=False)
strategy_comparison = pd.DataFrame([
    {'system': 'single_agent', 'score': baseline_strategy_score, **baseline_dimensions},
    {'system': 'crewai', 'score': crew_strategy_score, **crew_dimensions},
    {'system': 'single_agent_self_review', 'score': self_review_score, **self_review_dimensions},
])
strategy_comparison.to_csv(OUT_DIR / 'project3_strategy_comparison.csv', index=False)

project3_summary = {
    'model': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'router_schema_sha256': fingerprint(decision_schema(LANGCHAIN_TOOLS)),
    'router_prompt_sha256': fingerprint(ROUTER_INSTRUCTION),
    'knowledge_base_sha256': fingerprint(KNOWLEDGE_BASE),
    'router_cases_sha256': fingerprint(AGENT_TESTS),
    'gpu': torch.cuda.get_device_name(0),
    'decoding': {'do_sample': False, 'router_max_new_tokens': 192, 'router_attempts': 2},
    'evaluation_version': 'upgrade-v1',
    'strategy_score_definition': 'Keyword coverage smoke proxy, NOT independent content quality',
    'aggregate_output_token_cap_per_arm': TOTAL_OUTPUT_BUDGET,
    'usage_by_arm': {'one_shot': baseline_usage, 'self_review': self_review_usage, 'crewai': crew_usage},
    'seed': SEED,
    'knowledge_base_files': len(kb_documents),
    'langchain_tools': list(LANGCHAIN_TOOLS),
    'mcp_tools': len(mcp_style_manifest['tools']),
    'local_agent': local_agent_metrics,
    'strategy_tasks': len(strategy_tasks),
    'strategy_cases_sha256': fingerprint(strategy_tasks),
    'aggregate_input_token_cap_per_arm': TOTAL_INPUT_BUDGET,
    'strategy_independent_content_ratings': None,
    'strategy_budget_note': 'Matched maximum token budgets, not equal actual compute. Record actual input/output use per case; no independent content scores yet.',
    'strategy': {
        'single_agent_score': baseline_strategy_score,
        'crewai_score': crew_strategy_score,
        'relative_improvement_percent': strategy_lift,
        'single_agent_seconds': baseline_strategy_seconds,
        'crewai_seconds': crew_strategy_seconds,
        'single_agent_dimensions': baseline_dimensions,
        'crewai_dimensions': crew_dimensions,
    },
    'local_generation_calls': local_generation_calls,
}
(OUT_DIR / 'project3_summary.json').write_text(json.dumps(project3_summary, indent=2))

display(pd.DataFrame([local_agent_metrics]).round(4))
display(strategy_comparison.round(4))
failures = agent_results[~agent_results['task_success']][
    ['id', 'task', 'planned_tool', 'answer']
].to_dict(orient='records')
print('Failed evaluation cases:', json.dumps(failures))
print("First-case keyword proxy change (%), undefined if baseline is zero:", strategy_lift)
samples = agent_results[['id', 'task', 'planned_tool', 'answer']].head(4).to_dict(orient='records')
print('PROJECT3_SUMMARY_JSON=' + json.dumps(project3_summary))
print('PROJECT3_SAMPLES_JSON=' + json.dumps(samples))
print('PROJECT3_CREW_MEMO=' + json.dumps(crew_strategy))
print('Saved Project 3 artifacts to:', OUT_DIR)

review_pack, review_key = strategy_review_pack(strategy_tasks, strategy_study, SEED)
(OUT_DIR / "blind_review.json").write_text(json.dumps(review_pack, indent=2))
(OUT_DIR / "review_key.json").write_text(json.dumps(review_key, indent=2))


                                         prompt_only  ...     paired_task_success_constrained_vs_prompt_only
0  {'test_cases': 15, 'errors': 0, 'routing_corre...  ...  {'mean_change': 0.26666666666666666, 'ci95': [...

[1 rows x 3 columns]
                     system    score  ...  safeguards  risks_and_mitigations
0              single_agent  96.4286  ...        0.75                    1.0
1                    crewai  75.0000  ...        0.25                    1.0
2  single_agent_self_review  96.4286  ...        0.75                    1.0

[3 rows x 9 columns]
Failed evaluation cases: [{"id": "s1", "task": "Who are the primary buyers?", "planned_tool": "request_clarification", "answer": "I could not validate an action. Please clarify the request and required details."}, {"id": "s2", "task": "Which integrations are supported?", "planned_tool": "request_clarification", "answer": "I could not validate an action. Please clarify the request and required details."}, {"id": "s2", "task": "